# Corrigé · Exercice 1.1 : l'agent de support en `StateGraph`

> Notebook autonome : exécutez-le de haut en bas, stack démarrée.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
# Données fictives de NordTech SA (src/helpdesk_ops/donnees.py) : à lire pour savoir ce que les outils peuvent trouver
from helpdesk_ops.donnees import INVENTAIRE, KB

print(f"{len(INVENTAIRE)} serveurs : {', '.join(INVENTAIRE)}")
for article in KB:
    print(f"  [{article['id']}] {article['titre']}")

## 1. Les outils

Points clés : docstring qui dit **quand** utiliser l'outil, exemple d'argument, normalisation de l'entrée, et **message d'erreur explicite** plutôt qu'un `None` (le modèle peut se corriger au tour suivant).

In [ ]:
from langchain_core.tools import tool


@tool(parse_docstring=True)
def statut_serveur(nom: str) -> str:
    """Retourne le statut (en ligne, hors ligne, dégradé) d'un serveur de l'inventaire NordTech.

    Args:
        nom: nom d'hôte exact en minuscules, par exemple 'srv-vpn-01'
    """
    return INVENTAIRE.get(nom.strip().lower(), f"inconnu : '{nom}' n'est pas dans l'inventaire")


@tool(parse_docstring=True)
def chercher_kb(question: str) -> str:
    """Cherche dans la base de connaissances du help-desk NordTech les procédures utiles pour un problème.

    Args:
        question: description courte du problème, par exemple 'configurer le VPN' ou 'Outlook plante'
    """
    # Recherche par mots communs : volontairement simple, le vrai RAG sur Qdrant arrive en J3
    mots = {m for m in sans_accents(question).lower().replace("-", " ").split() if len(m) > 3}
    scores = []
    for article in KB:
        texte = sans_accents(article["titre"] + " " + article["contenu"]).lower().replace("-", " ")
        score = sum(mot in texte for mot in mots)
        if score:
            scores.append((score, article))
    if not scores:
        return "Aucun article trouvé. Proposer d'ouvrir un ticket."
    meilleurs = sorted(scores, key=lambda s: s[0], reverse=True)[:2]
    return "\n".join(f"[{a['id']}] {a['titre']} : {a['contenu']}" for _, a in meilleurs)


print(chercher_kb.invoke({"question": "comment configurer le VPN ?"}))

## 2. Le graphe

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

OUTILS = [statut_serveur, chercher_kb]
llm_outils = llm.bind_tools(OUTILS)

PROMPT_SYSTEME = SystemMessage(
    "Tu es l'agent support N1 du help-desk de NordTech SA. "
    "Réponds toujours en français, en 3 phrases maximum. "
    "Utilise statut_serveur pour l'état d'un serveur et chercher_kb pour une procédure. "
    "N'invente jamais de procédure ni de nom de serveur : si les outils ne donnent rien, propose d'ouvrir un ticket."
)


def agent(state: MessagesState) -> dict:
    return {"messages": [llm_outils.invoke([PROMPT_SYSTEME, *state["messages"]])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(OUTILS))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
support = builder.compile()

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"```mermaid\n{support.get_graph().draw_mermaid()}\n```"))

## 3. `repondre`, tracée et bornée

Le repli sur `GraphRecursionError` est une **réponse utilisateur**, pas une trace d'erreur : l'utilisateur ne doit jamais voir une exception Python. L'erreur reste visible dans Langfuse (niveau `ERROR` sur la trace).

In [ ]:
from langgraph.errors import GraphRecursionError

INITIALES = "corrige"
REPLI = "Je n'ai pas pu traiter votre demande automatiquement : je la transmets à un technicien N1."


def repondre(question: str) -> str:
    """Pose une question à l'agent ; ne lève jamais d'exception vers l'utilisateur."""
    try:
        etat = support.invoke(
            {"messages": [HumanMessage(question)]},
            config={"callbacks": [langfuse_handler], "run_name": "ex11-support", "recursion_limit": 10},
        )
        return etat["messages"][-1].content
    except GraphRecursionError:
        return REPLI


QUESTIONS = [
    "Le serveur srv-erp-02 est-il en ligne ?",
    "Outlook plante dès que j'ouvre une pièce jointe, que faire ?",
    "Mon VPN ne marche plus : srv-vpn-01 est-il en ligne, et comment reconfigurer le client ?",
]

with propagate_attributes(session_id=f"ex11-{INITIALES}", tags=["stategraph", "ex11"]):
    for q in QUESTIONS:
        print(f"❓ {q}\n💬 {repondre(q)}\n")
langfuse.flush()

### [Code] Vérifier le repli sans attendre une vraie boucle

Une limite de 1 étape suffit à déclencher `GraphRecursionError` : c'est un **test** du chemin d'erreur, à garder dans une suite de tests.

In [ ]:
try:
    support.invoke({"messages": [HumanMessage("srv-mail-01 ?")]}, config={"recursion_limit": 1})
except GraphRecursionError:
    print("✅ GraphRecursionError bien levée ; repondre() renverrait :", REPLI)

## Ce qu'il fallait observer dans l'UI

- **Sessions → `ex11-corrige`** : 3 traces `ex11-support`, chacune avec l'arbre `agent → tools → agent`.
- **Question 3** : deux outils (`statut_serveur` **et** `chercher_kb`), souvent dans le **même** `AIMessage` (appels parallèles), parfois en deux tours selon le modèle. Les deux sont corrects ; le Gantt montre la différence de latence.
- **Hallucination** : comparez mot à mot la réponse et l'article renvoyé par `chercher_kb`. Lors des tests de préparation (granite4.1:8b, 20/10/2026), le modèle a **ajouté** un chemin de menu (« Fichier > Options > Gestionnaire de compléments ») absent de KB-004, et traduit MFA par « authentification à facture ». Plausible, fluide… et faux. Durcir le prompt aide, mais la vraie réponse est une **métrique** mesurée en J4 (fidélité aux sources).

## Pour aller plus loin

- Ajoutez un serveur inconnu (« srv-paie-09 ») : le message d'erreur de l'outil empêche-t-il l'invention ?
- `granite4.1:3b` (`MODEL_SMALL`) réussit-il la question 3 ? Comparez les traces.